# 02 · Ingest files and design useful chunks

**Level:** beginner → intermediate · **Time:** 40 minutes.

**Learning outcome:** load TXT/Markdown/PDF inputs, normalize cautiously, preserve provenance,
implement overlapping word chunks, and compare chunk sizes using measurable observations.
A chunk is a retrieval unit: it must be small enough to find and large enough to understand.

Everything required runs from repository data and temporary TXT/Markdown examples.
An optional PDF cell accepts your own local PDF. No API call is needed in this notebook.


In [ ]:
from pathlib import Path
import sys

# Works when Jupyter starts in the repository or in notebooks/.
ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / "rag_workshop").is_dir()), None)
if ROOT is None:
    raise RuntimeError("Open this notebook from the LangchainPractical project folder.")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from rag_workshop import (
    Document, load_documents, chunk_documents, TfidfIndex, DenseIndex,
    bm25_search, reciprocal_rank_fusion, context_text, ask_claude,
    rag_answer, show_hits, settings, offline_mode,
)
import pandas as pd
from IPython.display import display, Markdown

print("Mode:", "OFFLINE: Claude calls skipped" if offline_mode() else "LIVE: Claude calls use API credits")
print("Claude model:", settings().model)


## 1. Create two small, reproducible source files

We use a temporary directory so rerunning the exercise cannot overwrite your real documents.
The same loader works on a directory of your own files. Each document keeps a source path
and title; PDF documents additionally keep a page number.


In [ ]:
import tempfile
import re
from hashlib import sha256

demo_workspace = tempfile.TemporaryDirectory(prefix="rag_ingestion_")
demo_dir = Path(demo_workspace.name)
(demo_dir / "leave.txt").write_text(
    "Employees receive 24 days of annual paid leave.\n\n"
    "Up to 5 unused days may be carried forward through March 31.\n",
    encoding="utf-8",
)
(demo_dir / "remote.md").write_text(
    "# Remote work policy\n\n"
    "Employees may work remotely 3 days per week with manager approval.\n\n"
    "## Availability\nMaintain agreed team collaboration hours.\n",
    encoding="utf-8",
)
print("Temporary examples:", [path.name for path in sorted(demo_dir.iterdir())])


## 2. Implement loaders instead of hiding ingestion

The loader handles UTF-8 TXT/Markdown and text-based PDF pages. Markdown syntax is kept
because headings carry useful structure. An empty extracted PDF page is reported and skipped.
**Scanned PDFs need OCR**; `pypdf` cannot recognize text stored only as an image.
Tables, reading order, headers, and columns still need human inspection after extraction.


In [ ]:
def load_local_file(path):
    path = Path(path)
    suffix = path.suffix.lower()
    # Include the resolved path so equal filenames in different folders do not collide.
    file_id = sha256(str(path.resolve()).encode("utf-8")).hexdigest()[:12]
    metadata = {"source": str(path), "title": path.stem, "format": suffix.lstrip(".")}
    if suffix in {".txt", ".md"}:
        text = path.read_text(encoding="utf-8")
        return [Document(id=file_id, text=text, metadata=metadata)] if text.strip() else []
    if suffix == ".pdf":
        from pypdf import PdfReader
        result = []
        for page_number, page in enumerate(PdfReader(path).pages, start=1):
            text = page.extract_text() or ""
            if text.strip():
                result.append(Document(id=f"{file_id}:p{page_number}", text=text,
                                       metadata={**metadata, "page": page_number}))
            else:
                print(f"No text on {path.name}, page {page_number}; inspect for OCR needs.")
        return result
    raise ValueError(f"Unsupported file type: {suffix}")

loaded = [document for path in sorted(demo_dir.iterdir()) for document in load_local_file(path)]
display(pd.DataFrame([{"id": d.id, **d.metadata, "text": d.text} for d in loaded]))


## 3. Optional PDF exercise

Set `PDF_PATH` to a local PDF you are allowed to use, for example `ROOT / "data" / "policy.pdf"`.
Leaving it as `None` makes the notebook fully runnable without extra files. The loader does
not send the PDF anywhere. Before adding extracted text to a live RAG call, confirm that
it is appropriate to send that content to the model provider.


In [ ]:
PDF_PATH = None  # Example: ROOT / "data" / "policy.pdf"
if PDF_PATH is None:
    print("Optional PDF exercise skipped. TXT and Markdown examples are ready below.")
else:
    pdf_documents = load_local_file(PDF_PATH)
    print("Pages with extracted text:", len(pdf_documents))
    display(pd.DataFrame([{"page": d.metadata["page"], "characters": len(d.text),
                           "preview": d.text[:240]} for d in pdf_documents]))
    loaded.extend(pdf_documents)


## 4. Clean gently and preserve provenance

Normalize line endings and repeated horizontal spaces while preserving paragraphs.
Do not lowercase or remove punctuation from stored source text: amounts, code, and proper
names matter. A content hash helps identify exact duplicates, but not near-duplicates.
If duplicate content belongs to different owners, retain all provenance in a real pipeline.


In [ ]:
def clean_text(text):
    text = text.replace("\r\n", "\n").replace("\r", "\n")
    text = re.sub(r"[ \t]+", " ", text)
    return re.sub(r"\n{3,}", "\n\n", text).strip()

cleaned = []
fingerprints = set()
for document in loaded:
    text = clean_text(document.text)
    fingerprint = sha256(text.encode("utf-8")).hexdigest()
    if text and fingerprint not in fingerprints:
        fingerprints.add(fingerprint)
        cleaned.append(Document(id=document.id, text=text,
                                metadata={**document.metadata, "content_sha256": fingerprint}))
print("Loaded:", len(loaded), "Unique nonempty documents:", len(cleaned))
print(cleaned[0].text)


## 5. Implement a sliding word window

With a size of 20 words and overlap of 5, the window moves 15 words each time. Overlap
repeats boundary context but increases index size and duplicated evidence. Word counting
is easy to teach; it is **not token counting**, and model limits are in tokens.
Store start/end offsets and the parent source ID so a hit can be traced back to its origin.


In [ ]:
def word_chunks(document, size=20, overlap=5):
    if not 0 <= overlap < size:
        raise ValueError("Require 0 <= overlap < size.")
    words = document.text.split()
    chunks = []
    for start in range(0, len(words), size - overlap):
        end = min(start + size, len(words))
        chunks.append(Document(
            id=f"{document.id}:w{start}-{end}",
            text=" ".join(words[start:end]),
            metadata={**document.metadata, "source_id": document.id,
                      "word_start": start, "word_end": end},
        ))
        if end == len(words):
            break
    return chunks

small_chunks = [chunk for document in cleaned for chunk in word_chunks(document, size=20, overlap=5)]
display(pd.DataFrame([{"chunk_id": d.id, "words": len(d.text.split()),
                       "parent": d.metadata["source_id"], "text": d.text} for d in small_chunks]))


## 6. Make the boundary problem visible

This synthetic passage places a complete fact across a 20-word boundary. A retrieval hit
containing only half the fact may be insufficient. We measure whether **one chunk contains
the complete exact phrase**. This is a transparent teaching proxy, not a general relevance metric.


In [ ]:
fact = "Unused leave may be carried forward for five days until March thirty first"
prefix = " ".join(["context"] * 16)
suffix = " ".join(["background"] * 45)
boundary_doc = Document(id="boundary_demo", text=f"{prefix} {fact} {suffix}",
                        metadata={"source": "synthetic classroom example"})
boundary_rows = []
for size, overlap in [(20, 0), (20, 8), (40, 0), (40, 10), (80, 0)]:
    pieces = word_chunks(boundary_doc, size=size, overlap=overlap)
    indexed_words = sum(len(piece.text.split()) for piece in pieces)
    boundary_rows.append({
        "size": size, "overlap": overlap, "chunks": len(pieces),
        "indexed_words": indexed_words,
        "complete_fact_in_one_chunk": any(fact in piece.text for piece in pieces),
        "duplication_ratio": round(indexed_words / len(boundary_doc.text.split()), 2),
    })
display(pd.DataFrame(boundary_rows))


## 7. Compare chunking on the workshop corpus

Smaller chunks can isolate relevant statements; larger chunks preserve surrounding
explanation but may dilute matching. Use the same query and inspect both the top hit
and the amount of text. There is no universally optimal chunk size.


In [ ]:
documents = load_documents()
query = "How many leave days can carry forward through March 31?"
comparison = []
for size, overlap in [(25, 5), (60, 10), (120, 20)]:
    pieces = [chunk for document in documents for chunk in word_chunks(document, size, overlap)]
    top = TfidfIndex(pieces).search(query, k=1)[0]
    comparison.append({"size": size, "overlap": overlap, "total_chunks": len(pieces),
                       "top_source": top["metadata"]["source_id"],
                       "top_score": round(top["score"], 3), "top_text": top["text"]})
display(pd.DataFrame(comparison))
print("Do not compare raw scores as if they were calibrated across different indexes.")


## 8. Exercise: add a heading to every Markdown chunk

**Try first:** split a Markdown document at headings and prepend the heading to each
resulting section. This prevents a sentence such as "3 days per week" from losing its
subject. Keep the original source metadata and add a `heading` field.

The sample solution is intentionally simple; nested lists, fenced code blocks, and
complex Markdown need a proper parser. Compare its readable sections with fixed windows.


In [ ]:
def heading_sections(document):
    heading = document.metadata.get("title", "Document")
    buffer = []
    sections = []

    def emit():
        body = "\n".join(buffer).strip()
        if body:
            sections.append(Document(
                id=f"{document.id}:section{len(sections)}",
                text=f"{heading}\n{body}",
                metadata={**document.metadata, "source_id": document.id, "heading": heading},
            ))

    for line in document.text.splitlines():
        if re.match(r"^#{1,6}\s+", line):
            emit()
            heading = re.sub(r"^#{1,6}\s+", "", line).strip()
            buffer = []
        else:
            buffer.append(line)
    emit()
    return sections

markdown_document = next(document for document in cleaned if document.metadata["format"] == "md")
sections = heading_sections(markdown_document)
display(pd.DataFrame([{"heading": d.metadata["heading"], "text": d.text} for d in sections]))


## 9. Check that your chunks remain traceable

These small checks validate data invariants, not retrieval quality. Empty chunks and lost
source IDs are pipeline bugs that can be caught before embedding or API calls.


In [ ]:
assert small_chunks and all(chunk.text.strip() for chunk in small_chunks)
assert len({chunk.id for chunk in small_chunks}) == len(small_chunks)
assert all(chunk.metadata.get("source") and chunk.metadata.get("source_id") for chunk in small_chunks)
assert all(len(chunk.text.split()) <= 20 for chunk in small_chunks)
print("Checks passed: nonempty, unique IDs, source metadata, and size bounds.")
# Temporary examples are now in memory; remove their disposable source directory.
demo_workspace.cleanup()


## Instructor demo and next step

Change overlap to zero and explain the boundary table. Then increase it close to chunk size
and discuss why duplication becomes expensive. Add a second Markdown heading and rerun the
section exercise. For your own PDF, inspect at least two extracted pages before indexing it.

**Checkpoint:** explain the difference between a source document, a chunk, and a token;
show where the source path, page number, and parent ID survive.

Next: **03_embeddings_and_semantic_search.ipynb** converts chunks to vectors.

Official references: [pypdf text extraction](https://pypdf.readthedocs.io/en/stable/user/extract-text.html),
[Python pathlib](https://docs.python.org/3/library/pathlib.html),
[Python tempfile](https://docs.python.org/3/library/tempfile.html).
